# Parallel & Concurrent Programming

## Measure before optimizing

The `duplicates` function below returns the items that appear several times in a list.

1. With `timeit.timeit`, measure the execution time of `duplicates(numbers)`.
2. With `cProfile.run`, find the call in which the function spends most of its time.
3. Write a faster version, check that it returns the same items and measure the gain with `timeit`.

**Useful documentation**

- [`timeit.timeit`](https://docs.python.org/3/library/timeit.html#timeit.timeit)
- [`cProfile.run`](https://docs.python.org/3.13/library/profile.html#profile.run)
- [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter)

In [ ]:
import random

numbers = [random.randrange(5_000) for _ in range(5_000)]


def duplicates(values: list[int]) -> list[int]:
  result = []
  for value in values:
    if values.count(value) > 1 and value not in result:
      result.append(value)
  return result


# Your code here

### Additional hints

`timeit.timeit("duplicates(numbers)", globals=globals(), number=3)` returns the total duration of the 3 runs, in seconds: divide it by 3. `globals=globals()` gives the measured code access to the notebook's variables. `cProfile.run("duplicates(numbers)", sort="tottime")` prints a table sorted by the time spent in each function itself: look at the first line. A fast version counts every occurrence in a single pass, with `collections.Counter`.

```python
duration = timeit.timeit("duplicates(numbers)", globals=globals(), number=3) / 3
print(f"duplicates: {duration:.3f}s")

cProfile.run("duplicates(numbers)", sort="tottime")


def fast_duplicates(values: list[int]) -> list[int]:
  return [value for value, n in collections.Counter(values).items() if ...]
```

### Solution

In [ ]:
import collections
import cProfile
import timeit

duration = timeit.timeit("duplicates(numbers)", globals=globals(), number=3) / 3
print(f"duplicates: {duration:.3f}s")

cProfile.run("duplicates(numbers)", sort="tottime")


def fast_duplicates(values: list[int]) -> list[int]:
  return [value for value, n in collections.Counter(values).items() if n > 1]


assert sorted(fast_duplicates(numbers)) == sorted(duplicates(numbers))
duration = timeit.timeit("fast_duplicates(numbers)", globals=globals(), number=3) / 3
print(f"fast_duplicates: {duration:.5f}s")

The profile shows that almost all the time is spent in `list.count`, called for every item: each call scans the whole list again, so the algorithm is quadratic. [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter) counts every occurrence in a single pass.

## Computation: Threads or Processes?

The `count_primes(start, stop)` function below counts the prime numbers in a range: it is pure computation, without input/output.

Count the prime numbers from 0 to 400,000, split into 4 chunks (`chunks`), in three ways, measuring how long each takes with `time.perf_counter`:

1. sequentially;
2. with a `concurrent.futures.ThreadPoolExecutor` of 4 threads;
3. with a `concurrent.futures.ProcessPoolExecutor` of 4 processes: the same code, where only the class changes.

The three results must be equal. How do you explain the durations you get?

**Useful documentation**

- [`time.perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter)

In [ ]:
import multiprocessing

# The processes created below must find the functions defined in this notebook.
# "fork" copies them along with it (Linux and macOS); "forkserver", Linux's
# default since Python 3.14, doesn't find them and the pool hangs.
multiprocessing.set_start_method("fork", force=True)


def count_primes(start: int, stop: int) -> int:
  count = 0
  for n in range(max(start, 2), stop):
    if all(n % d for d in range(2, int(n**0.5) + 1)):
      count += 1
  return count


chunks = [(i, i + 100_000) for i in range(0, 400_000, 100_000)]

In [ ]:
# Your code here

### Additional hints

`executor.map(count_primes, starts, stops)` calls `count_primes(starts[0], stops[0])`, `count_primes(starts[1], stops[1])`… like `zip`: first split the `chunks` into their starts and their stops. The result is an iterator, which `sum` consumes. `max_workers=4` sets the number of threads or processes. Measure each version between two calls to `time.perf_counter()`.

```python
starts = [a for a, _ in chunks]
stops = [b for _, b in chunks]

start_time = time.perf_counter()
sequential = sum(count_primes(a, b) for a, b in chunks)
print(f"Sequential: {time.perf_counter() - start_time:.2f}s")

start_time = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
  threads = sum(executor.map(...))
print(f"Threads: {time.perf_counter() - start_time:.2f}s")
```

### Solution

In [ ]:
import concurrent.futures
import time

starts = [a for a, _ in chunks]
stops = [b for _, b in chunks]

start_time = time.perf_counter()
sequential = sum(count_primes(a, b) for a, b in chunks)
print(f"Sequential: {time.perf_counter() - start_time:.2f}s")

start_time = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
  threads = sum(executor.map(count_primes, starts, stops))
print(f"Threads: {time.perf_counter() - start_time:.2f}s")

start_time = time.perf_counter()
with concurrent.futures.ProcessPoolExecutor(max_workers=4) as executor:
  processes = sum(executor.map(count_primes, starts, stops))
print(f"Processes: {time.perf_counter() - start_time:.2f}s")

print(sequential == threads == processes)

The threads are no faster than the sequential version: the GIL lets only one thread run Python code at a time (except in the *free-threaded* build of Python, with no GIL, supported since Python 3.14). Processes each have their own interpreter and really compute in parallel, within the number of cores (2 on Colab). Switching from threads to processes only took changing the executor's class.

To go further: on your machine, rerun this computation with `python3.14t` (the *free-threaded* build): threads compute in parallel there.

## Fixing an Argument: `functools.partial`

`Pool.map` calls its function with a single argument; the `f` function below takes two.

1. With `functools.partial`, call `f` in parallel in a `multiprocessing.Pool()`, for `n` from 0 to 9 and with `verbose=True`. How many different processes did the work? Compare with `os.cpu_count()`, the number of processes `Pool()` creates by default.
2. Now try `pool.map(lambda n: f(n, False), range(10))`. What error do you get? Why does `partial` work, and not the `lambda`?

**Useful documentation**

- [`functools.partial`](https://docs.python.org/3/library/functools.html#functools.partial)
- [`os.cpu_count`](https://docs.python.org/3/library/os.html#os.cpu_count)

In [ ]:
import os


def f(n: int, verbose: bool) -> int:
  if verbose:
    print(f"n = {n}, process {os.getpid()} (parent {os.getppid()})")
  return n * 2


# Your code here

### Additional hints

`functools.partial(f, verbose=True)` returns a new one-argument function, which calls `f(n, verbose=True)`: that's what `pool.map` receives. The numbers printed after "process" are the PIDs of the pool's processes. For the `lambda`, wrap the call in a `try` / `except Exception as error`, and print `type(error).__name__` and `error`.

```python
with multiprocessing.Pool() as pool:
  results = pool.map(functools.partial(f, ...), range(10))
print(results)
print(os.cpu_count())

with multiprocessing.Pool() as pool:
  try:
    pool.map(lambda n: f(n, False), range(10))
  except Exception as error:
    print(...)
```

### Solution

In [ ]:
import functools
import multiprocessing
import os

with multiprocessing.Pool() as pool:
  results = pool.map(functools.partial(f, verbose=True), range(10))
print(results)
print(f"Default number of pool processes: {os.cpu_count()}")

with multiprocessing.Pool() as pool:
  try:
    pool.map(lambda n: f(n, False), range(10))
  except Exception as error:
    print(f"{type(error).__name__}: {error}")

`Pool.map` sends each task to one of the pool's processes by serializing it with `pickle`. `pickle` records a function by its name, which the process looks up in the module where it is defined: a `lambda` has no name, hence the `PicklingError`. A `partial` object, on the other hand, is serialized as the function `f` (by its name) and its fixed arguments.

`Pool()` creates `os.cpu_count()` processes, but hands them the tasks in batches (`chunksize`): tasks as short as these are often all handled by just a few processes.

## Performing Two Different Tasks in Parallel with a `Pool`

The following code submits two different computations to the same `Pool` with [`map_async`](https://docs.python.org/3/library/multiprocessing.html#multiprocessing.pool.Pool.map_async), which returns immediately instead of waiting for the results like `map`: both computations therefore run at the same time on the pool's processes.

Rewrite it with [`concurrent.futures.ProcessPoolExecutor`](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.ProcessPoolExecutor).

In [ ]:
import multiprocessing


def a(i: int) -> int:
  return i * 2


def b(i: int) -> int:
  return i**2


if __name__ == "__main__":
  with multiprocessing.Pool() as pool:
    a_results = pool.map_async(a, range(10))
    b_results = pool.map_async(b, range(10))
    print(a_results.get())
    print(b_results.get())

In [ ]:
# Your code here

### Additional hints

Replace `multiprocessing.Pool()` with `concurrent.futures.ProcessPoolExecutor()`. `executor.submit(a, i)` submits a computation without waiting and immediately returns a `Future`; `future.result()` then waits for its result. Submit all the computations of `a` and `b` before reading any result, so that they run at the same time.

```python
import concurrent.futures

if __name__ == "__main__":
  with concurrent.futures.ProcessPoolExecutor() as executor:
    a_futures = [executor.submit(a, i) for i in range(10)]
    b_futures = ...
    print([future.result() for future in a_futures])
    print(...)
```

### Solution

In [ ]:
import concurrent.futures


def a(i: int) -> int:
  return i * 2


def b(i: int) -> int:
  return i**2


if __name__ == "__main__":
  with concurrent.futures.ProcessPoolExecutor() as executor:
    a_futures = [executor.submit(a, i) for i in range(10)]
    b_futures = [executor.submit(b, i) for i in range(10)]
    print([future.result() for future in a_futures])
    print([future.result() for future in b_futures])

## Creating a Producer/Consumer Architecture

The following code makes two processes communicate through a [`multiprocessing.Queue`](https://docs.python.org/3/library/multiprocessing.html#multiprocessing.Queue): the producer puts items in it, the consumer takes them out and prints them.

Run it, then answer: why does the producer put `None` at the end? Then modify the code to start two consumers instead of one.

In [ ]:
import multiprocessing


def consumer(queue: multiprocessing.Queue) -> None:
  while True:
    item = queue.get()
    if item is None:
      break
    print(item)


def producer(queue: multiprocessing.Queue) -> None:
  for i in range(10):
    queue.put(i)
  queue.put(None)


if __name__ == "__main__":
  queue = multiprocessing.Queue()
  consumer_process = multiprocessing.Process(target=consumer, args=(queue,))
  producer_process = multiprocessing.Process(target=producer, args=(queue,))
  consumer_process.start()
  producer_process.start()
  consumer_process.join()
  producer_process.join()

In [ ]:
# Your code here

### Additional hints

The consumer only leaves its loop when it receives `None`. With two consumers, each must receive its own: the producer puts as many `None` as there are consumers. Give each consumer a name to see which one prints what.

```python
def producer(queue: multiprocessing.Queue, n_consumers: int) -> None:
  for i in range(10):
    queue.put(i)
  for _ in range(...):
    queue.put(None)
```

### Solution

`None` is a sentinel value: it tells the consumer that no more items will come. Without it, the consumer would block forever on `queue.get()`. With several consumers, each one must receive its own sentinel:

In [ ]:
import multiprocessing


def consumer(name: str, queue: multiprocessing.Queue) -> None:
  while True:
    item = queue.get()
    if item is None:
      break
    print(f"{name}: {item}")


def producer(queue: multiprocessing.Queue, n_consumers: int) -> None:
  for i in range(10):
    queue.put(i)
  for _ in range(n_consumers):
    queue.put(None)


if __name__ == "__main__":
  queue = multiprocessing.Queue()
  consumers = [
    multiprocessing.Process(target=consumer, args=(f"consumer {k}", queue))
    for k in range(2)
  ]
  producer_process = multiprocessing.Process(
    target=producer, args=(queue, len(consumers))
  )
  for process in [*consumers, producer_process]:
    process.start()
  for process in [*consumers, producer_process]:
    process.join()

## A Shared Account Between Threads: `threading.Lock`

Four threads credit the same account, 1,000 times each: they read the balance, wait for a moment (an input/output, simulated by `time.sleep(0)`), then write the balance plus 1.

1. Write this `credit()` function, run it in 4 `threading.Thread`s, wait for them, then print `balance`. Is it 4,000?
2. Fix `credit` with a `threading.Lock` held around the read and the write (`with lock:`), and check the balance.

In [ ]:
import threading
import time

balance = 0

# Your code here

### Additional hints

`global balance` lets the function modify the notebook's variable. Between the read and the write, `time.sleep(0)` hands over to another thread, which reads the same balance. Without this wait, `balance += 1` alone almost never shows the error in CPython 3.13, without being safe for all that. The lock lets only one thread at a time into the `with lock:` block. Reset `balance` to 0 before the second version.

```python
def credit() -> None:
  global balance
  for _ in range(1_000):
    value = balance
    time.sleep(0)
    balance = ...


threads = [threading.Thread(target=credit) for _ in range(4)]
for thread in threads:
  thread.start()
for thread in threads:
  thread.join()
print(balance)
```

### Solution

In [ ]:
import threading
import time


def credit() -> None:
  global balance
  for _ in range(1_000):
    value = balance
    time.sleep(0)
    balance = value + 1


def credit_with_lock() -> None:
  global balance
  for _ in range(1_000):
    with lock:
      value = balance
      time.sleep(0)
      balance = value + 1


def run_threads(target) -> None:
  threads = [threading.Thread(target=target) for _ in range(4)]
  for thread in threads:
    thread.start()
  for thread in threads:
    thread.join()


balance = 0
run_threads(credit)
print(f"Without a lock: {balance}")

balance = 0
lock = threading.Lock()
run_threads(credit_with_lock)
print(f"With a lock: {balance}")

Without a lock, a thread reads the balance, hands over during `time.sleep(0)`, and another one reads the same value: each then writes that value plus 1, and one of the two credits is lost. This is a race condition. With the lock, the read and the write happen as one block, with no other thread in between: the balance is 4,000. The lock also makes these credits sequential: only the critical section goes inside it, as short as possible.

## Downloading Several Pages: Sequential or Threads?

Write a function `fetch(url, path)` that downloads `url` into the file `path`, then download the pages of `to_download` sequentially, then with a `concurrent.futures.ThreadPoolExecutor` (`map` method), measuring how long each version takes.

To download a page, you can use the following code, where `url` is the URL to download and `path` the path of the file to write:

```python
request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
with urllib.request.urlopen(request) as response, path.open("wb") as fh:
  shutil.copyfileobj(response, fh)
```

The `User-Agent` header is required: websites such as Python Tutor or Wikipedia reject the default `urllib` agent (HTTP error 403). `USER_AGENT` says who makes the request and how to reach them, as their policies ask.

Why do threads speed up these downloads, when they did not speed up counting prime numbers?

**Useful documentation**

- [`urllib.request.urlopen`](https://docs.python.org/3/library/urllib.request.html#urllib.request.urlopen)
- [`shutil.copyfileobj`](https://docs.python.org/3/library/shutil.html#shutil.copyfileobj)

In [ ]:
import pathlib

USER_AGENT = "shuuchuu-labs/1.0 (https://github.com/shuuchuu/labs)"

downloads = pathlib.Path("downloads")
downloads.mkdir(exist_ok=True)

to_download = (
  ("https://docs.python.org/fr/3/", downloads / "python-docs.html"),
  ("https://pythontutor.com/", downloads / "python-tutor.html"),
  ("https://www.google.com/", downloads / "google.html"),
  ("https://peps.python.org/pep-0008/", downloads / "pep-8.html"),
  ("https://pypi.org/", downloads / "pypi.html"),
)

# Your code here

### Additional hints

`executor.map(fetch, urls, paths)` calls `fetch(urls[0], paths[0])`, `fetch(urls[1], paths[1])`…: first split `to_download` into two lists. `map` returns an iterator: `list(...)` waits for the downloads to end and raises their errors.

```python
def fetch(url: str, path: pathlib.Path) -> None: ...


start = time.perf_counter()
for url, path in to_download:
  fetch(url, path)
print(f"Sequential: {time.perf_counter() - start:.2f}s")

urls = [url for url, _ in to_download]
paths = [path for _, path in to_download]
start = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor() as executor:
  list(executor.map(...))
print(f"Threads: {time.perf_counter() - start:.2f}s")
```

### Solution

In [ ]:
import concurrent.futures
import pathlib
import shutil
import time
import urllib.request


def fetch(url: str, path: pathlib.Path) -> None:
  request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
  with urllib.request.urlopen(request) as response, path.open("wb") as fh:
    shutil.copyfileobj(response, fh)


start = time.perf_counter()
for url, path in to_download:
  fetch(url, path)
print(f"Sequential: {time.perf_counter() - start:.2f}s")

urls = [url for url, _ in to_download]
paths = [path for _, path in to_download]
start = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor() as executor:
  list(executor.map(fetch, urls, paths))
print(f"Threads: {time.perf_counter() - start:.2f}s")

A thread waiting for the network releases the GIL: the downloads' waits overlap. Threads suit input/output, processes suit computation.

## Concurrent downloads with `asyncio`

Reuse `to_download` and the `fetch` function from the previous exercise, and download the pages concurrently with `asyncio`: `asyncio.to_thread` runs a blocking function (such as `fetch`) in a thread, without blocking the event loop, and an `asyncio.TaskGroup` starts several tasks and waits for all of them. Display the total download time.

Beware: in Colab (as in Jupyter), an event loop is already running, so `asyncio.run(main())` raises a `RuntimeError`. Write `await main()` directly in the cell; in a script, you would use `asyncio.run(main())`.

**Useful documentation**

- [`asyncio.to_thread`](https://docs.python.org/3/library/asyncio-task.html#asyncio.to_thread)

In [ ]:
# Your code here

### Additional hints

`asyncio.to_thread(fetch, url, path)` returns a coroutine; `tg.create_task(...)` starts it as a task of the group. The `async with asyncio.TaskGroup() as tg:` block only ends once all its tasks are done. Then run `await main()`.

```python
async def main() -> None:
  start = time.perf_counter()
  async with asyncio.TaskGroup() as tg:
    for url, path in to_download:
      tg.create_task(...)
  print(f"{len(to_download)} pages downloaded in {time.perf_counter() - start:.2f}s")
```

### Solution

In [ ]:
import asyncio
import time


async def main() -> None:
  start = time.perf_counter()
  async with asyncio.TaskGroup() as tg:
    for url, path in to_download:
      tg.create_task(asyncio.to_thread(fetch, url, path))
  print(f"{len(to_download)} pages downloaded in {time.perf_counter() - start:.2f}s")


await main()

`asyncio.to_thread` still runs `fetch` in threads: the event loop only waits for them to end, and the gain comes from the threads, as in the previous exercise. `asyncio.gather(*(asyncio.to_thread(fetch, url, path) for url, path in to_download))` would do the same; `TaskGroup` (Python 3.11) is recommended for new code: if a task raises an exception, it cancels the others, and the errors come up together in an `ExceptionGroup`.

The point of `asyncio` is non-blocking input/output, without threads: an asynchronous library such as `httpx` (preinstalled on Colab) makes the requests in the event loop itself. To go further, try this version, then `await main_httpx()`:

```python
import httpx


async def fetch_async(
  client: httpx.AsyncClient, url: str, path: pathlib.Path
) -> None:
  response = await client.get(url, follow_redirects=True)
  path.write_bytes(response.content)


async def main_httpx() -> None:
  async with httpx.AsyncClient(headers={"User-Agent": USER_AGENT}) as client:
    async with asyncio.TaskGroup() as tg:
      for url, path in to_download:
        tg.create_task(fetch_async(client, url, path))
```